# 🩺 Digital Twin for Type 2 Diabetes — Notebook 1: Building Our Virtual Patients

**What this notebook does:** it creates 100 realistic *synthetic* (computer-generated) patients with Type 2 Diabetes.
Every patient gets two kinds of data, exactly as the competition asks:

1. **Static / historical data (EHR)** — who the patient is: age, BMI, lab results (HbA1c, cholesterol), medicines, past diagnoses, and a genetic marker.
2. **Dynamic / real-time data (wearables)** — 14 days of readings every 5 minutes: continuous glucose monitor (CGM), heart rate, steps, sleep stages and logged meals.

**Why synthetic data?** Real patient data is protected by India's DPDP Act. The competition *requires* synthetic or anonymized data, so we build our own —
but we build it using **real medical relationships** so the patients behave like real people.

**How to run:** click each grey code cell and press **Shift + Enter**, top to bottom. Read the text cells — they explain what's happening.

## Step 0 — Setup
We load three tools: **numpy** (maths), **pandas** (tables, like Excel inside Python) and **matplotlib** (charts).
We also fix a "random seed" so that everyone who runs this notebook gets exactly the same patients. This makes our work **reproducible** — judges love that.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

N_PATIENTS = 100          # how many virtual patients
N_DAYS = 14               # how many days of wearable data per patient
STEP_MIN = 5              # a CGM gives one reading every 5 minutes
SLOTS_PER_DAY = 24 * 60 // STEP_MIN   # = 288 readings per day
START_DATE = pd.Timestamp("2026-03-02")  # a Monday

print(f"We will create {N_PATIENTS} patients x {N_DAYS} days x {SLOTS_PER_DAY} readings/day "
      f"= {N_PATIENTS * N_DAYS * SLOTS_PER_DAY:,} wearable readings")

## Step 1 — The Electronic Health Record (EHR)

This is what a doctor sees in the hospital file. Key terms in plain language:

- **BMI** (Body Mass Index): weight relative to height. For Indians, 23+ is considered overweight (lower cut-off than Western countries).
- **HbA1c**: a blood test showing *average* blood sugar over the last ~3 months. Below 5.7% is normal, 6.5%+ means diabetes, and above 8% means poorly controlled.
- **TCF7L2 risk alleles** (0, 1 or 2): the strongest known *genetic* risk marker for Type 2 Diabetes. Everyone has two copies of this gene; each "risky" copy raises risk.
- **Medications**: Metformin is the standard first medicine. Doctors add a second drug (DPP-4 or SGLT2 inhibitor) or insulin when sugar stays high.
- **Blood pressure, cholesterol (LDL/HDL), triglycerides, creatinine**: standard labs — diabetes often comes with high BP and cholesterol, and can damage kidneys (creatinine).

The numbers are generated with **medically sensible links**: e.g. higher BMI and longer disease duration → higher HbA1c; higher HbA1c → stronger medication.

In [ ]:
def generate_ehr(n, rng):
    rows = []
    for i in range(n):
        age = int(rng.integers(35, 76))
        sex = rng.choice(["Male", "Female"])
        bmi = float(np.clip(rng.normal(26.5, 4.0), 18.5, 40))
        years_with_diabetes = float(np.clip(rng.gamma(2.0, 3.0), 0.5, 25))
        tcf7l2 = int(rng.binomial(2, 0.30))              # genetic risk alleles
        family_history = bool(rng.random() < 0.45 + 0.10 * tcf7l2)

        hba1c = float(np.clip(6.3 + 0.06 * (bmi - 26) + 0.09 * years_with_diabetes
                              + 0.15 * tcf7l2 + rng.normal(0, 0.8), 5.9, 11.5))

        # Doctors escalate treatment as HbA1c rises
        if hba1c < 6.5:
            medication = rng.choice(["Lifestyle only", "Metformin"], p=[0.4, 0.6])
        elif hba1c < 8.0:
            medication = "Metformin"
        elif hba1c < 9.5:
            medication = rng.choice(["Metformin + DPP-4 inhibitor", "Metformin + SGLT2 inhibitor"])
        else:
            medication = rng.choice(["Metformin + Basal insulin", "Metformin + SGLT2 inhibitor"], p=[0.6, 0.4])

        systolic_bp = float(np.clip(118 + 0.45 * (age - 35) + 0.8 * (bmi - 24) + rng.normal(0, 10), 95, 190))
        diastolic_bp = float(np.clip(0.55 * systolic_bp + rng.normal(8, 6), 60, 115))
        ldl = float(np.clip(rng.normal(115, 30), 50, 220))
        hdl = float(np.clip(rng.normal(44 if sex == "Male" else 50, 8) - 0.3 * (bmi - 24), 25, 80))
        triglycerides = float(np.clip(rng.lognormal(np.log(140 + 5 * (bmi - 24)), 0.3), 60, 500))
        creatinine = float(np.clip(rng.normal(0.95 if sex == "Male" else 0.75, 0.15)
                                   + 0.004 * (age - 35) + 0.01 * years_with_diabetes, 0.5, 2.2))

        diagnoses = ["Type 2 Diabetes"]
        if systolic_bp >= 140 or rng.random() < 0.10:
            diagnoses.append("Hypertension")
        if ldl >= 130 or triglycerides >= 200:
            diagnoses.append("Dyslipidemia")
        if bmi >= 27.5:
            diagnoses.append("Obesity")
        if years_with_diabetes > 10 and rng.random() < 0.3:
            diagnoses.append("Peripheral neuropathy")

        rows.append(dict(
            patient_id=f"P{i + 1:03d}", age=age, sex=sex, bmi=round(bmi, 1),
            years_with_diabetes=round(years_with_diabetes, 1), hba1c_pct=round(hba1c, 1),
            tcf7l2_risk_alleles=tcf7l2, family_history_diabetes=family_history,
            medication=medication, systolic_bp=round(systolic_bp), diastolic_bp=round(diastolic_bp),
            ldl_mg_dl=round(ldl), hdl_mg_dl=round(hdl), triglycerides_mg_dl=round(triglycerides),
            creatinine_mg_dl=round(creatinine, 2), past_diagnoses="; ".join(diagnoses),
            activity_level=rng.choice(["Low", "Moderate", "High"], p=[0.45, 0.40, 0.15]),
            diet_pattern=rng.choice(["Rice-dominant", "Wheat-dominant", "Mixed"], p=[0.40, 0.35, 0.25]),
        ))
    return pd.DataFrame(rows)

ehr = generate_ehr(N_PATIENTS, rng)
ehr.head()

## Step 2 — Each patient's "hidden body"

Two people can eat the same plate of rice and get very different sugar spikes. We give every patient **hidden physiology settings** —
the model will *never* see these directly. It has to learn them from the data, just like a doctor learns a patient over time.
That is the heart of a digital twin: **learning how one specific body behaves.**

- **Insulin sensitivity**: how fast the body clears sugar from the blood. Lower with high BMI, high HbA1c and long diabetes; medicines improve it.
- **Carb sensitivity**: how much sugar rises per gram of carbohydrate eaten.
- **Dawn phenomenon**: an early-morning rise in sugar caused by hormones (cortisol, growth hormone) — very common in diabetes.
- **Sleep quality & walking habit**: lifestyle traits.
- **Target average glucose (eAG)**: from the real ADAG formula `eAG = 28.7 × HbA1c − 46.7`. We calibrate each patient's glucose so their 14-day average matches their HbA1c — this keeps the EHR and wearable data **consistent with each other**.

In [ ]:
MED_BOOST = {"Lifestyle only": 1.00, "Metformin": 1.15,
             "Metformin + DPP-4 inhibitor": 1.25, "Metformin + SGLT2 inhibitor": 1.30,
             "Metformin + Basal insulin": 1.40}

def hidden_physiology(p, rng):
    impairment = 1 + 0.04 * (p.bmi - 22) + 0.12 * (p.hba1c_pct - 6) + 0.02 * p.years_with_diabetes
    return dict(
        insulin_sensitivity=0.022 / impairment * MED_BOOST[p.medication] * rng.lognormal(0, 0.15),
        carb_sensitivity=2.8 * (1 + 0.10 * (p.hba1c_pct - 6)) * rng.lognormal(0, 0.15),
        dawn_amplitude=max(0.0, rng.normal(8 + 4 * (p.hba1c_pct - 6), 5)),
        sleep_trait=rng.choice(["Good", "Fair", "Poor"], p=[0.40, 0.35, 0.25]),
        walk_habit={"Low": 0.10, "Moderate": 0.30, "High": 0.60}[p.activity_level] * rng.uniform(0.6, 1.4),
        resting_hr=float(np.clip(64 + 0.35 * (p.bmi - 22) - {"Low": 0, "Moderate": 2, "High": 6}[p.activity_level]
                                 + rng.normal(0, 4), 52, 92)),
        target_mean_glucose=28.7 * p.hba1c_pct - 46.7,
    )

hidden = {p.patient_id: hidden_physiology(p, rng) for p in ehr.itertuples()}
pd.DataFrame(hidden).T.head()

## Step 3 — Simulating 14 days of daily life

For each patient and each day we simulate a realistic Indian routine:

- **Sleep**: bedtime around 11 pm, later on weekends. "Poor sleepers" sleep less and get less deep sleep. Each night is split into **sleep stages** (Light, Deep, REM, brief Awake) following ~90-minute sleep cycles — deep sleep early in the night, REM later, like real smartwatches report.
- **Meals**: breakfast, a heavy lunch (the biggest carb load, especially for rice-dominant diets), an evening chai-and-snack, and a late dinner (~9 pm, common in India). Weekends are bigger, and ~7% of days are "celebration days" with sweets.
  Patients **log** most meals in an app — but they forget some and misjudge portion sizes (±20%), just like real people.
- **Steps**: light movement through the day, random walking bouts, optional morning walks, and **post-meal walks** (which are known to blunt sugar spikes).
- **Stress / sick days**: ~5% of days, when the body handles sugar worse.

In [ ]:
def gamma_absorption(carbs_g, tau_min, n_slots=48):
    """How carbs enter the blood over ~4 hours after a meal (grams per 5-min slot)."""
    t = np.arange(n_slots) * STEP_MIN + STEP_MIN / 2
    return carbs_g * t / tau_min ** 2 * np.exp(-t / tau_min) * STEP_MIN

def slot(day, hour):
    return int(round((day * 24 + hour) * 60 / STEP_MIN))

def simulate_lifestyle(p, h, rng):
    T = N_DAYS * SLOTS_PER_DAY
    carb_abs = np.zeros(T + 60)          # carbs entering the blood, per slot
    carbs_logged = np.zeros(T)
    steps = np.zeros(T)
    stage = np.array([""] * T, dtype=object)
    nights = []

    sleep_mean = {"Good": 7.3, "Fair": 6.5, "Poor": 5.6}[h["sleep_trait"]]
    deep_scale = {"Good": 1.0, "Fair": 0.85, "Poor": 0.65}[h["sleep_trait"]]
    awake_p = {"Good": 0.03, "Fair": 0.06, "Poor": 0.10}[h["sleep_trait"]]
    tau_diet = {"Rice-dominant": 30, "Wheat-dominant": 40, "Mixed": 35}[p.diet_pattern]
    base_steps = {"Low": 6, "Moderate": 11, "High": 16}[p.activity_level]
    morning_walk_p = {"Low": 0.08, "Moderate": 0.30, "High": 0.60}[p.activity_level]

    # ---- Sleep: night n runs from evening of day n to morning of day n+1 ----
    wake_hours = {}
    for n in range(-1, N_DAYS):
        weekend = (START_DATE + pd.Timedelta(days=n + 1)).dayofweek >= 5
        duration = float(np.clip(rng.normal(sleep_mean + (0.5 if weekend else 0), 0.8), 3.5, 10))
        bed = 23.0 + rng.normal(0, 0.6) + (0.4 if weekend else 0)
        wake = bed + duration - 24                       # hour on next morning
        wake_hours[n + 1] = wake
        s0, s1 = slot(n, bed), slot(n, bed + duration)
        awake_slots, deep_slots, total = 0, 0, 0
        for s in range(max(s0, 0), min(s1, T)):
            m = (s - s0) * STEP_MIN
            cycle, pos = m // 90, (m % 90) / 90
            deep_w = max(0.0, 0.55 - 0.13 * cycle) * deep_scale if 0.1 < pos < 0.6 else 0.02
            rem_w = min(0.6, 0.20 + 0.08 * cycle) if pos >= 0.6 else 0.03
            r = rng.random()
            if r < awake_p:
                st = "Awake"
            elif r < awake_p + deep_w:
                st = "Deep"
            elif r < awake_p + deep_w + rem_w:
                st = "REM"
            else:
                st = "Light"
            stage[s] = st
            total += 1
            awake_slots += st == "Awake"
            deep_slots += st == "Deep"
        if n + 1 < N_DAYS:
            nights.append(dict(day=n + 1, sleep_hours=duration,
                               sleep_efficiency=1 - awake_p if total == 0 else 1 - awake_slots / total,
                               deep_pct=0 if total == 0 else deep_slots / total))

    asleep = stage != ""

    # ---- Meals, activity, stress ----
    stress = np.ones(N_DAYS)
    for d in range(N_DAYS):
        weekend = (START_DATE + pd.Timedelta(days=d)).dayofweek >= 5
        celebration = rng.random() < 0.07
        if rng.random() < 0.05:
            stress[d] = rng.uniform(0.70, 0.85)
        size = (1.15 if weekend else 1.0) * (1.10 if p.diet_pattern == "Rice-dominant" else 1.0)
        wake = wake_hours[d]

        meals = [
            (max(wake + 0.5, rng.normal(8.2 + (0.7 if weekend else 0), 0.5)), rng.uniform(35, 70), True),
            (rng.normal(13.5, 0.7), rng.uniform(60, 115), True),
            (rng.normal(17.5, 0.6), rng.uniform(15, 40), rng.random() < 0.65),   # chai + snack
            (rng.normal(21.0, 0.7), rng.uniform(55, 105), True),
        ]
        if celebration:
            meals.append((rng.normal(19.0, 1.0), rng.uniform(30, 60), True))    # sweets
        for i, (hour, carbs, happens) in enumerate(meals):
            if not happens:
                continue
            carbs *= size
            s = slot(d, hour)
            tau = tau_diet * rng.uniform(0.8, 1.25) * (0.8 if i == 4 else 1.0)
            carb_abs[s:s + 48] += gamma_absorption(carbs, tau)
            if rng.random() < 0.85:                                              # patient logs the meal
                carbs_logged[s] += round(carbs * rng.uniform(0.8, 1.2))
            if i in (1, 3) and rng.random() < h["walk_habit"]:                    # post-meal walk
                w0 = s + int(rng.integers(2, 6))
                steps[w0:w0 + int(rng.integers(3, 7))] += rng.uniform(350, 550)

        if rng.random() < morning_walk_p:
            w0 = slot(d, wake + rng.uniform(0.3, 1.0))
            steps[w0:w0 + int(rng.integers(5, 10))] += rng.uniform(400, 600)

    awake_day = ~asleep
    steps += np.where(awake_day, rng.poisson(base_steps, T), 0)
    bouts = awake_day & (rng.random(T) < 0.008)
    for s in np.where(bouts)[0]:
        steps[s:s + int(rng.integers(2, 5))] += rng.uniform(300, 500)
    steps[asleep] = 0

    return dict(carb_abs=carb_abs[:T], carbs_logged=carbs_logged, steps=np.round(steps),
                stage=stage, asleep=asleep, nights=pd.DataFrame(nights), stress=stress)

## Step 4 — The glucose engine (the physiology)

This is a simplified version of how blood sugar really works. Every 5 minutes:

`new sugar = old sugar + (sugar arriving from food) − (sugar the body clears)`

How fast the body clears sugar depends on:
- the patient's **insulin sensitivity** (from Step 2),
- **recent activity** — muscles pull sugar out of the blood when you walk,
- **last night's sleep** — research shows even one short night makes the body more insulin-resistant the next day,
- **stress / sick days**.

On top of that we add the **dawn phenomenon**, small slow natural fluctuations, and finally **CGM sensor noise and gaps**
(real sensors are slightly noisy and sometimes lose signal) — so our data needs cleaning, just like real data.

In [ ]:
def simulate_glucose(p, h, life, rng):
    T = N_DAYS * SLOTS_PER_DAY
    hours = (np.arange(T) * STEP_MIN / 60) % 24
    day = np.arange(T) // SLOTS_PER_DAY

    sleep_by_day = life["nights"].set_index("day")["sleep_hours"].reindex(range(N_DAYS)).fillna(7).values
    sleep_factor = 1 - 0.06 * np.clip(7 - sleep_by_day, 0, None)                 # short sleep → resistance
    recent_steps = pd.Series(life["steps"]).rolling(6, min_periods=1).sum().values  # last 30 min
    activity = np.clip(recent_steps / 2500, 0, 1)

    clearance = h["insulin_sensitivity"] * sleep_factor[day] * life["stress"][day] * (1 + 1.2 * activity)
    rise = h["carb_sensitivity"] * life["carb_abs"]

    x = np.zeros(T)
    for t in range(1, T):
        x[t] = x[t - 1] + rise[t] - STEP_MIN * clearance[t] * x[t - 1]

    dawn = h["dawn_amplitude"] * np.exp(-0.5 * ((hours - 6.5) / 1.0) ** 2)
    drift = np.zeros(T)
    for t in range(1, T):
        drift[t] = 0.99 * drift[t - 1] + rng.normal(0, 1.2)

    shape = x + dawn + drift
    baseline = max(80.0, h["target_mean_glucose"] - shape.mean())            # calibrate to HbA1c
    true_glucose = np.clip(baseline + shape, 40, 400)

    cgm = true_glucose + rng.normal(0, 4, T)                                    # sensor noise
    cgm[rng.random(T) < 0.01] = np.nan                                          # random dropouts
    for _ in range(int(rng.integers(0, 3))):                                    # longer signal gaps
        g0 = int(rng.integers(0, T - 40))
        cgm[g0:g0 + int(rng.integers(6, 36))] = np.nan
    return np.round(cgm, 1), true_glucose, baseline

## Step 5 — Heart rate and heart rate variability (HRV)

- **Heart rate** drops during sleep (lowest in deep sleep), rises with walking and slightly after meals, and is higher after a bad night.
- **HRV (RMSSD)** is the tiny variation between heartbeats, measured overnight by smartwatches. **Higher HRV = healthier, more relaxed nervous system.**
  It falls with age, obesity, poor sugar control and bad sleep. Low HRV is a known early warning sign in diabetes (it can signal nerve damage to the heart).

In [ ]:
def simulate_heart(p, h, life, rng):
    T = N_DAYS * SLOTS_PER_DAY
    day = np.arange(T) // SLOTS_PER_DAY
    stage = life["stage"]
    nights = life["nights"].set_index("day").reindex(range(N_DAYS))
    poor_night = (nights["sleep_hours"].fillna(7).values < 6).astype(float)

    hr = np.full(T, h["resting_hr"] + 8.0)
    hr[stage == "Light"] = h["resting_hr"] - 5
    hr[stage == "REM"] = h["resting_hr"] - 2
    hr[stage == "Deep"] = h["resting_hr"] - 9
    hr[stage == "Awake"] = h["resting_hr"]
    hr += np.clip(life["steps"] / 500 * 28, 0, 45)
    hr += 3 * np.clip(pd.Series(life["carb_abs"]).values / 3, 0, 1)
    hr += 3 * poor_night[day] + rng.normal(0, 2.5, T)
    hr = np.round(np.clip(hr, 42, 180))

    base_hrv = 48 - 0.45 * (p.age - 35) - 0.6 * (p.bmi - 22) - 1.5 * (p.hba1c_pct - 6) \
               - 0.3 * p.years_with_diabetes
    base_hrv = max(12.0, base_hrv + rng.normal(0, 4))
    n = life["nights"]
    hrv = base_hrv * np.sqrt(n["sleep_hours"] / 7.5) * (0.85 + 0.3 * n["deep_pct"] / 0.2).clip(0.8, 1.2) \
          * rng.lognormal(0, 0.12, len(n))
    return hr, np.round(hrv.values, 1)

## Step 6 — Run the simulation for all 100 patients

This takes about 30–60 seconds. We produce three tables (saved as CSV files, which open in Excel too):

1. `ehr_patients.csv` — one row per patient (the hospital record)
2. `wearable_timeseries.csv` — one row every 5 minutes per patient (CGM, heart rate, steps, sleep stage, logged carbs)
3. `daily_summary.csv` — one row per patient per day (sleep hours, sleep quality, overnight HRV, total steps) — like the daily summary screen in Apple Health / Google Fit

In [ ]:
ts_frames, daily_frames, truth = [], [], {}
timestamps = pd.date_range(START_DATE, periods=N_DAYS * SLOTS_PER_DAY, freq=f"{STEP_MIN}min")

for p in ehr.itertuples():
    h = hidden[p.patient_id]
    life = simulate_lifestyle(p, h, rng)
    cgm, true_glucose, baseline = simulate_glucose(p, h, life, rng)
    hr, hrv = simulate_heart(p, h, life, rng)
    truth[p.patient_id] = true_glucose

    ts_frames.append(pd.DataFrame(dict(
        patient_id=p.patient_id, timestamp=timestamps, cgm_glucose_mg_dl=cgm,
        heart_rate_bpm=hr, steps=life["steps"].astype(int),
        sleep_stage=np.where(life["stage"] == "", "Not asleep", life["stage"]),
        carbs_logged_g=life["carbs_logged"].astype(int))))

    n = life["nights"]
    daily_steps = life["steps"].reshape(N_DAYS, SLOTS_PER_DAY).sum(axis=1)
    daily_frames.append(pd.DataFrame(dict(
        patient_id=p.patient_id, date=[(START_DATE + pd.Timedelta(days=int(d))).date() for d in n["day"]],
        sleep_hours=n["sleep_hours"].round(2), sleep_efficiency_pct=(100 * n["sleep_efficiency"]).round(1),
        deep_sleep_pct=(100 * n["deep_pct"]).round(1), overnight_hrv_rmssd_ms=hrv,
        total_steps=daily_steps[n["day"].values].astype(int))))

wearables = pd.concat(ts_frames, ignore_index=True)
daily = pd.concat(daily_frames, ignore_index=True)

# Fasting glucose in the EHR = average glucose between 6 and 7 am (consistent with wearable data)
fasting = wearables[wearables.timestamp.dt.hour == 6].groupby("patient_id").cgm_glucose_mg_dl.mean()
ehr["fasting_glucose_mg_dl"] = ehr.patient_id.map(fasting).round().astype(int)

ehr.to_csv("ehr_patients.csv", index=False)
wearables.to_csv("wearable_timeseries.csv", index=False)
daily.to_csv("daily_summary.csv", index=False)
print("Saved:", ehr.shape, wearables.shape, daily.shape)

## Step 7 — Sanity checks: do our patients behave like real ones?

A good team doesn't just generate data — it **proves the data is realistic**. We check against known clinical facts:

- **Time in Range (TIR)**: % of time glucose stays between 70–180 mg/dL. International guidelines say TIR ≈ 70% corresponds to HbA1c ≈ 7%. Worse HbA1c → lower TIR.
- The CGM average should match the eAG formula from HbA1c.
- Daily steps should look like typical Indian adults with diabetes (roughly 3,000–10,000).

👉 These checks go straight into our presentation as "data validation".

In [ ]:
g = wearables.groupby("patient_id").cgm_glucose_mg_dl
check = ehr[["patient_id", "hba1c_pct", "medication"]].copy()
check["cgm_mean"] = check.patient_id.map(g.mean()).round(1)
check["eAG_from_hba1c"] = (28.7 * check.hba1c_pct - 46.7).round(1)
check["time_in_range_pct"] = check.patient_id.map(g.apply(lambda s: ((s >= 70) & (s <= 180)).mean() * 100)).round(1)
check["time_above_180_pct"] = check.patient_id.map(g.apply(lambda s: (s > 180).mean() * 100)).round(1)

print("Average Time-in-Range by HbA1c band:")
print(check.groupby(pd.cut(check.hba1c_pct, [5.8, 6.5, 7.0, 8.0, 9.0, 12]), observed=True)
      [["time_in_range_pct", "time_above_180_pct"]].mean().round(1))
print("\nCorrelation HbA1c vs Time-in-Range:", round(check.hba1c_pct.corr(check.time_in_range_pct), 2), "(should be strongly negative)")
print("Median daily steps:", int(daily.total_steps.median()))
print("Median sleep hours:", round(daily.sleep_hours.median(), 1))
print(f"Missing CGM readings: {wearables.cgm_glucose_mg_dl.isna().mean() * 100:.1f}% (realistic sensor gaps)")

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(check.hba1c_pct, check.time_in_range_pct, alpha=0.7)
ax.axhline(70, ls="--", c="green"); ax.axvline(7, ls="--", c="green")
ax.set_xlabel("HbA1c (%) from EHR"); ax.set_ylabel("Time in Range 70–180 (%) from CGM")
ax.set_title("EHR and wearable data agree: higher HbA1c → less time in range")
plt.tight_layout(); plt.show()

## Step 8 — Meet one virtual patient

Let's look at two days of one patient. Look for: sugar rising after every meal (orange lines), the big lunch/dinner spikes,
the red dashed line at 180 mg/dL (our "spike" threshold), the heart rate dipping at night, and walking bursts.

In [ ]:
def plot_patient(pid, days=2, start_day=3):
    d = wearables[wearables.patient_id == pid]
    d = d[(d.timestamp >= START_DATE + pd.Timedelta(days=start_day)) &
          (d.timestamp < START_DATE + pd.Timedelta(days=start_day + days))]
    info = ehr[ehr.patient_id == pid].iloc[0]
    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    axes[0].plot(d.timestamp, d.cgm_glucose_mg_dl, lw=1.5)
    axes[0].axhline(180, c="red", ls="--", label="Spike threshold 180")
    axes[0].axhspan(70, 180, color="green", alpha=0.07, label="Target range")
    for t, c in d[d.carbs_logged_g > 0][["timestamp", "carbs_logged_g"]].values:
        axes[0].axvline(t, c="orange", alpha=0.6)
        axes[0].text(t, axes[0].get_ylim()[1] * 0.97, f"{c}g", fontsize=8, color="darkorange")
    axes[0].set_ylabel("Glucose (mg/dL)"); axes[0].legend(loc="upper left", fontsize=8)
    axes[0].set_title(f"{pid}: {info.age}y {info.sex}, BMI {info.bmi}, HbA1c {info.hba1c_pct}%, {info.medication}")
    axes[1].plot(d.timestamp, d.heart_rate_bpm, c="crimson", lw=1); axes[1].set_ylabel("Heart rate")
    axes[2].bar(d.timestamp, d.steps, width=0.003, color="teal"); axes[2].set_ylabel("Steps / 5 min")
    plt.tight_layout(); plt.show()

plot_patient("P001")

## Step 9 — Download the data
Run this cell to download the three CSV files to your Mac. Keep them in one folder called `data` — we'll put them on GitHub later.
(In Colab you can also click the 📁 folder icon on the left to see them.)

In [ ]:
try:
    from google.colab import files
    for f in ["ehr_patients.csv", "daily_summary.csv", "wearable_timeseries.csv"]:
        files.download(f)
except ImportError:
    print("Not running in Colab — files are saved in the current folder.")